In [1]:
#import libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import tensorflow as tf
from tensorflow.keras import layers
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import wandb
from wandb.integration.keras import WandbCallback
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from wandb.sdk.data_types.graph import Graph



I0000 00:00:1775723779.750588  649343 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
#Les problèmes sont wandb et save best model dans le cas 2

In [13]:
# Task 1.1.1 - regular ANN

In [2]:
early_stop = EarlyStopping(
    monitor='val_loss',   # surveille la loss sur validation
    patience=3,           # attend 3 epochs sans amélioration
    restore_best_weights=True
)

In [8]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # divise le LR par 2
    patience=2,      # après 2 epochs sans progrès
    min_lr=1e-5
)

In [3]:
checkpoint = ModelCheckpoint(
    "best_model.keras",          # nom du fichier
    monitor="val_loss",       # on surveille la validation
    save_best_only=True,      # garde seulement le meilleur
    mode="min",               # plus petite loss = mieux
    verbose=1
)

In [14]:
# Charger le fichier (IMPORTANT : separator = tab)
df = pd.read_csv("amazon_cells_labelled_LARGE_25K.txt", sep="\t", header=None)

# Renommer colonnes
df.columns = ["text", "label"]

# Séparer
X = df["text"]
y = df["label"]

# Train + temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Validation + test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(len(X_train), len(X_val), len(X_test))


print(X.head())
print(y.head())

17500 3750 3750
0    I've read this book with much expectation, it ...
1    love it...very touch.it's to bad that in the d...
2    The creepiest book I've ever read! It's a cree...
3    It starts off a bit slow, but once the product...
4    As good as this book may be, the print quality...
Name: text, dtype: str
0    0
1    1
2    1
3    1
4    0
Name: label, dtype: int64


In [15]:
# Compter le nombre de 0 et de 1
counts = df['label'].value_counts()
print("Nombre de 1 :", counts.get(1, 0))
print("Nombre de 0 :", counts.get(0, 0))

Nombre de 1 : 15116
Nombre de 0 : 9884


In [16]:
counts = df['label'].value_counts()
percent = counts / counts.sum() * 100
print(percent)

label
1    60.464
0    39.536
Name: count, dtype: float64


In [5]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_val_vec   = vectorizer.transform(X_val).toarray()
X_test_vec  = vectorizer.transform(X_test).toarray()

In [6]:
model_ann = tf.keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(5000,)),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1775723794.400614  649343 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8254 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:21:00.0, compute capability: 7.5


In [7]:
model_ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [8]:
wandb.init(
    project="Lab1",
    name="NN_model_tf",
    config={
        "learning_rate": 0.0001,
        "batch_size": 32,
        "epochs": 20,
        "early_stopping": early_stop,
        "dropout": 0.5
    }
)

config = wandb.config

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [9]:
model_ann.fit(
    X_train_vec, y_train,
    validation_data=(X_val_vec, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[
        early_stop,
        checkpoint,
        WandbCallback(log_model=False,save_graph=False,save_model=False)  
    ]
)
wandb.finish()

wandb: WARNING WandbCallback is deprecated and will be removed in a future release. Please use the WandbMetricsLogger, WandbModelCheckpoint, and WandbEvalCallback callbacks instead. See https://docs.wandb.ai/guides/integrations/keras for more information.


Epoch 1/20


I0000 00:00:1775723805.747570  649619 service.cc:153] XLA service 0x7f9dbc032b20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775723805.747597  649619 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5 (Driver: 13.0.0; Runtime: 12.3.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1775723805.785414  649619 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775723805.955655  649619 cuda_dnn.cc:461] Loaded cuDNN version 91900
I0000 00:00:1775723805.998394  649619 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1430__.10


 45/547 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6013 - loss: 0.6804

I0000 00:00:1775723807.445085  649619 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


541/547 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7351 - loss: 0.4981

I0000 00:00:1775723809.782290  649619 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1430__.10


547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7360 - loss: 0.4970
Epoch 1: val_loss improved from None to 0.29139, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
547/547 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8137 - loss: 0.3936 - val_accuracy: 0.8717 - val_loss: 0.2914
Epoch 2/20
534/547 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9063 - loss: 0.2385
Epoch 2: val_loss did not improve from 0.29139
547/547 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9035 - loss: 0.2422 - val_accuracy: 0.8725 - val_loss: 0.2982
Epoch 3/20
537/547 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9349 - loss: 0.1738
Epoch 3: val_loss did not improve from 0.29139
547/547 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9299 - loss: 0.1840 - val_accuracy: 0.8717 - val_loss: 0.3170
Epoch 4/20
530/547 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9532 - loss: 0.1338
Epoch 4: val_loss did not improve from 0.29139
547/547 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

accuracy,▁▆▇█
epoch,▁▃▆█
loss,█▄▂▁
val_accuracy,▇█▇▁
val_loss,▁▂▄█
accuracy,0.95234
best_epoch,0
best_val_loss,0.29139
epoch,3
loss,0.13595
val_accuracy,0.86827


In [10]:
best_model = load_model("best_model.keras")
y_pred = best_model.predict(X_val_vec)
y_pred = (y_pred > 0.5).astype(int)

118/118 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [11]:
cm = confusion_matrix(y_val, y_pred)
print(cm)

[[1267  202]
 [ 279 2002]]


In [12]:
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1469
           1       0.91      0.88      0.89      2281

    accuracy                           0.87      3750
   macro avg       0.86      0.87      0.87      3750
weighted avg       0.87      0.87      0.87      3750



In [ ]:
#Task 1.1.2 - LSTM

In [3]:
# Charger le fichier (IMPORTANT : separator = tab)
df = pd.read_csv("amazon_cells_labelled_LARGE_25K.txt", sep="\t", header=None)

# Renommer colonnes
df.columns = ["text", "label"]

# Séparer
X = df["text"]
y = df["label"]

# Train + temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Validation + test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
# Paramètres
MAX_VOCAB = 10000
MAX_LEN = 100  # longueur max des phrases

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)  # X_train = liste de phrases

# Séquences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

# Padding pour avoir la même longueur
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')

In [4]:
checkpoint = ModelCheckpoint(
    "best_model_LSTM.keras",          # nom du fichier
    monitor="val_loss",       # on surveille la validation
    save_best_only=True,      # garde seulement le meilleur
    mode="min",               # plus petite loss = mieux
    verbose=1
)

In [5]:
VOCAB_SIZE = MAX_VOCAB
EMBED_DIM = 128
LSTM_UNITS = 64
DROPOUT_RATE = 0.5

model_bilstm = Sequential()
model_bilstm.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM))  # plus besoin de input_length
model_bilstm.add(Bidirectional(LSTM(LSTM_UNITS)))
model_bilstm.add(Dropout(DROPOUT_RATE))
model_bilstm.add(Dense(1, activation='sigmoid'))

model_bilstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_bilstm.summary()

I0000 00:00:1775722488.388529  645467 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 8254 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:21:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Initialise WandB
wandb.init(project="Lab1", name="BiLSTM_sentiment_analysis_TF")

# Entraînement
model_bilstm.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[
        early_stop,        # ton callback EarlyStopping
        lr_scheduler,      # ton callback LearningRateScheduler
        checkpoint,        # sauvegarde du meilleur modèle
        WandbCallback(log_model=False,save_graph=False,save_model=False)  # graphe désactivé ici pour éviter l'erreur
    ]
)

wandb.finish()

accuracy,▁
epoch,▁
learning_rate,▁
loss,▁
val_accuracy,▁
val_loss,▁
accuracy,0.81754
best_epoch,0
best_val_loss,0.29739
epoch,0
learning_rate,0.001


Epoch 1/20
545/547 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.9181 - loss: 0.2160
Epoch 1: val_loss improved from 0.29739 to 0.29259, saving model to best_model_LSTM.keras

Epoch 1: finished saving model to best_model_LSTM.keras
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.9125 - loss: 0.2285 - val_accuracy: 0.8784 - val_loss: 0.2926 - learning_rate: 0.0010
Epoch 2/20
547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9460 - loss: 0.1563
Epoch 2: val_loss did not improve from 0.29259
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.9403 - loss: 0.1634 - val_accuracy: 0.8741 - val_loss: 0.3241 - learning_rate: 0.0010
Epoch 3/20
546/547 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9635 - loss: 0.1130
Epoch 3: val_loss did not improve from 0.29259
547/547 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - accuracy: 0.9578 - loss: 0.1229 - val_accuracy: 0.8683 - val_loss: 0.3633 - learning_rate: 0.0010
Epoch 4/20
546/547 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.97

accuracy,▁▄▆█
epoch,▁▃▆█
learning_rate,███▁
loss,█▅▃▁
val_accuracy,█▆▃▁
val_loss,▁▂▄█
accuracy,0.97411
best_epoch,0
best_val_loss,0.29259
epoch,3
learning_rate,0.0005
